# ImageJ Data Analysis

#Upload/Data frame/Configuration

## Import data

In [ ]:
import pandas as pd
from google.colab import files
import io

print("Please upload your CSV data files. These files will be used to create the main DataFrame 'df'.")

uploaded = files.upload()

# Initialize an empty list to store individual DataFrames
all_dfs = []

# Loop through each uploaded file and read it into a DataFrame
for name, content in uploaded.items():
    # Read CSV, using the first column as index (assuming it's a row number)
    temp_df = pd.read_csv(io.BytesIO(content), index_col=0)
    all_dfs.append(temp_df)

# Concatenate all DataFrames into a single DataFrame
if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    print("Files uploaded successfully and DataFrame 'df' created/updated.")
    # --- Debugging: Print df info after creation ---
    print("\nDEBUG: df.info() after file upload and concatenation:")
    df.info()
    print("\nDEBUG: df['Label'].value_counts() after file upload and concatenation:")
    display(df['Label'].value_counts())
    # ------------------------------------------------
else:
    print("No files were uploaded. DataFrame 'df' was not created or updated.")

## Create dataframe

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Removed: from mpl_toolkits.mplot3d import Axes3D # Import for 3D plotting
#from google.colab import drive
from google.colab import files
import io
import re
import numpy as np # For histogram2d

# Define global parameters for consistency and easy modification
DIAMETER_MIN_FILTER = None # Set to a number (e.g., 55) to filter out particles below this diameter, or None to disable
EXCLUDE_CONDITIONS = [] # List of conditions to exclude from analysis. Set to [] to include all.
EXCLUDE_DAYS = [] # List of days to exclude from analysis. Set to [] to include all.

# Define the desired order for 'Condition' categories
CONDITION_ORDER = ['DS', 'HS', 'PEG', 'PVA', 'DSPEG', 'DSPVA', 'HSPEG', 'NEG'] #

# The DataFrame 'df' is now expected to be created by a preceding cell (e.g., file upload cell).
# This section, previously responsible for conditional file upload, has been removed
# to ensure 'df' consistently uses the uploaded data.

# Drop the 'Feret' column as requested by the user
if 'Feret' in df.columns:
    df = df.drop(columns=['Feret'])

# 2. Clean the Label column by stripping the file extension (.jpg, .tif, etc.)
# These replacements might not be necessary if the 'Label' column from the CSV
# does not contain file extensions or ' (X)' patterns, but kept for robustness.
df['Clean_Label'] = df['Label'].str.replace(r'\.[a-zA-Z0-9]+$', '', regex=True)

# New: Remove the ' (X)' part (e.g., ' (3)') often created by duplicate uploads.
# This standardizes the label before splitting for metadata parsing.
df['Clean_Label'] = df['Clean_Label'].str.replace(r' \(\d+\)$', '', regex=True)

# --- Added for debugging: Check Clean_Label values before parsing ---
print("DEBUG: Clean_Label value counts before metadata parsing:")
display(df['Clean_Label'].value_counts())
# -------------------------------------------------------------------

# 3. Parse the metadata from the file name
# Filename format: BATCH_PLATE_DAY_Condition_DATE_Magnification_Combined_Results.csv (or similar)
def parse_filename_metadata(clean_label):
    # Preprocess clean_label to standardize 'DayX' to 'DAYX' (case-insensitive)
    clean_label = re.sub(r'(D|d)ay(\d+)', r'DAY\2', clean_label)
    # Updated regex to explicitly capture Condition as alphanumeric/hyphen,
    # and properly allow for optional DATE and Magnification fields
    # Corrected \d to \d for digit matching.
    # Modified Magnification regex to [\d.]+x to allow for decimal points (e.g., '2.0x')
    match = re.match(r'(.+?)_(.+?)_(DAY\d+)_([A-Z0-9-]+)(?:_(\d{4}-\d{2}-\d{2}))?(?:_([\d.]+x))?.*', clean_label)

    if match:
        data = {
            'BATCH': match.group(1),
            'PLATE': match.group(2),
            'DAY': match.group(3),
            'DATE': match.group(5),  # DATE is now group 5 if present
            'Magnification': match.group(6) # Magnification is now group 6 if present
        }

        # Extract the raw condition string (e.g., 'DS-01-021' or 'DSPEG-SAMPLE')
        raw_condition = match.group(4)

        # Initialize found_condition to None
        found_condition = None

        # Sort CONDITION_ORDER by length in descending order to match more specific conditions first
        sorted_condition_order = sorted(CONDITION_ORDER, key=len, reverse=True)

        for ordered_cond in sorted_condition_order:
            # Check for exact match or match followed by a non-alphanumeric separator
            if raw_condition == ordered_cond or \
               (raw_condition.startswith(ordered_cond) and \
                len(raw_condition) > len(ordered_cond) and \
                not raw_condition[len(ordered_cond)].isalnum()):
                found_condition = ordered_cond
                break

        # If a matching condition from CONDITION_ORDER is found, use it; otherwise, use the raw string
        data['Condition'] = found_condition if found_condition else raw_condition

        # Debugging: print for first few calls to inspect parsing results with final assigned values
        # Reset call_count for successful parses to allow more failure messages to show
        if not hasattr(parse_filename_metadata, 'success_call_count'):
            parse_filename_metadata.success_call_count = 0
        if parse_filename_metadata.success_call_count < 10:
            print(f"DEBUG: Parsed '{clean_label}' -> BATCH: {data['BATCH']}, PLATE: {data['PLATE']}, DAY: {data['DAY']}, Condition (Final): {data['Condition']}, DATE: {data['DATE']}, Magnification: {data['Magnification']}")
            parse_filename_metadata.success_call_count += 1

        # Explicitly return Series with correct column order to ensure proper assignment
        return pd.Series(
            [data['BATCH'], data['PLATE'], data['DAY'], data['Condition'], data['DATE'], data['Magnification']],
            index=['BATCH', 'PLATE', 'DAY', 'Condition', 'DATE', 'Magnification']
        )
    else:
        # Debugging: print for all failed calls
        print(f"DEBUG: Failed to parse '{clean_label}'")
        # Return default or NaN for non-matching patterns
        return pd.Series({
            'BATCH': None, 'PLATE': None, 'DAY': None,
            'Condition': None, 'DATE': None, 'Magnification': None
        })

# Apply the parsing function to create new columns
df[['BATCH', 'PLATE', 'DAY', 'Condition', 'DATE', 'Magnification']] = df['Clean_Label'].apply(parse_filename_metadata)

# Debugging: Check df state after metadata parsing
print(f"DEBUG: DataFrame rows after metadata parsing: {len(df)}")
print(f"DEBUG: df.head() after metadata parsing:")
display(df.head())
print(f"DEBUG: df.info() after metadata parsing:")
df.info()


# Convert 'Condition' to a categorical type with the specified order
df['Condition'] = pd.Categorical(df['Condition'], categories=CONDITION_ORDER, ordered=True)

# If there were other filename patterns (e.g., 'DayX_TreatmentA_Results.csv'),
# more sophisticated logic would be needed here to extract varying conditions.

# Apply filters with .copy() to prevent SettingWithCopyWarning
df_original_rows_count_before_filters = len(df)
if EXCLUDE_CONDITIONS:
    df = df[~df['Condition'].isin(EXCLUDE_CONDITIONS)].copy()
if EXCLUDE_DAYS:
    df = df[~df['DAY'].isin(EXCLUDE_DAYS)].copy()

print(f"DEBUG: DataFrame rows after EXCLUDE filters: {len(df)}")
print(f"DEBUG: Unique DAY values before final dropna: {df['DAY'].unique()}")
print(f"DEBUG: Number of non-null DAY values before final dropna: {df['DAY'].count()}")

# It's good practice to filter out rows from df where parsing failed for 'DAY'
# as these rows would cause issues in analyses grouped by 'DAY'.
df_rows_before_day_dropna = len(df)
df = df.dropna(subset=['DAY']).copy() # Added .copy() here too

if len(df) == 0:
    print("Error: All data rows were dropped because the 'DAY' metadata could not be parsed or after filtering.")
elif len(df) < df_rows_before_day_dropna:
    print(f"Warning: Dropped {df_rows_before_day_dropna - len(df)} rows due to missing 'DAY' metadata.")
print(f"DEBUG: DataFrame rows after DAY dropna: {len(df)}")

# --- Added for debugging: Check BATCH distribution before summary_df creation ---
print("DEBUG: BATCH value counts before summary_df creation:")
display(df['BATCH'].value_counts())
# ----------------------------------------------------------------------------------

# If df is empty after all processing, create an empty summary_df to prevent errors
if df.empty:
    print("DEBUG: DataFrame is empty after all filtering and parsing. Cannot calculate summary_df.")
    summary_df = pd.DataFrame(columns=['BATCH', 'PLATE', 'DAY', 'Condition', 'Particle count', 'Dia_mean', 'Dia_median', 'Dia_std', 'Circ._mean', 'Circ._median', 'Circ._std'])
else:
    # Calculate Diameter from Area assuming circular particles
    df['Diameter'] = 2 * np.sqrt(df['Area'] / np.pi)

    # Calculate Circularity from Area and Perimeter
    df['Circ.'] = (4 * np.pi * df['Area']) / (df['Perim.']**2)

    # Filter out particles with Diameter less than the defined minimum, if enabled
    if DIAMETER_MIN_FILTER is not None:
        df = df[df['Diameter'] > DIAMETER_MIN_FILTER].copy()

    # 4. Create summary dataframe (Group by BATCH, DAY, Condition)
    summary_df = df.groupby(['BATCH','PLATE' ,'DAY', 'Condition'], observed=False).agg({
        'Diameter': ['mean', 'median', 'std', 'count'], # Added count for Particle count, removed Area
        'Circ.': ['mean', 'median', 'std']
    }).reset_index()


    # 1. Get unique days present in the data (assuming 'DayX' format)
    # Filter out None values before getting unique days
    valid_days = df['DAY'].dropna().unique()

    # 2. Sort them naturally (Day2, Day4, Day5, Day10...)
    # This regex extracts the digits and sorts by the integer value
    def natural_sort_key(s):
        # This function now expects s to not be NaN, so the check can be simplified
        return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', str(s))]

    # Only sort if there are valid days to sort
    if len(valid_days) > 0:
        sorted_days = sorted(valid_days, key=natural_sort_key)
    else:
        sorted_days = [] # No valid days, so no categories

    # 3. Convert the DAY column to a categorical type with this specific order
    # Ensure sorted_days is not empty before converting to categorical
    if sorted_days:
        df['DAY'] = pd.Categorical(df['DAY'], categories=sorted_days, ordered=True)
    else:
        print("Warning: No valid 'DAY' categories found after parsing. 'DAY' column will not be converted to categorical and subsequent analysis relying on it might be affected.")

    # Flatten the multi-level column names for cleaner viewing
    summary_df.columns = ['_'.join(col).strip('_') for col in summary_df.columns.values]

    # Rename columns
    summary_df = summary_df.rename(columns={
        'Diameter_count': 'Particle count',
        'Diameter_mean': 'Dia_mean',
        'Diameter_median': 'Dia_median',
        'Diameter_std': 'Dia_std'

    })

    # Reorder columns to include new metadata fields and Circularity
    cols = ['BATCH', 'DAY', 'Condition', 'Particle count',
            'Dia_mean', 'Dia_median', 'Dia_std',
            'Circ._mean', 'Circ._median', 'Circ._std']
    summary_df = summary_df[cols]

    # Format numerical columns to 1 decimal place
    for col in summary_df.columns:
        if summary_df[col].dtype == 'float64' or summary_df[col].dtype == 'int64':
            summary_df[col] = summary_df[col].round(1)

# Display the summary dataframe
print("--- Summary DataFrame ---")
display(summary_df)

In [ ]:
#BATCH_ORDER = ['STD-MAN-R5', 'STD-ON-R5', 'ZEI-ON-R5'] # Corrected batch names (from R5 back to R6)
#df['BATCH'] = pd.Categorical(df['BATCH'], categories=BATCH_ORDER, ordered=True)
#summary_df['BATCH'] = pd.Categorical(summary_df['BATCH'], categories=BATCH_ORDER, ordered=True)
#print(f"'BATCH' column converted to categorical with order: {BATCH_ORDER}")

## Configuration Parameters

In [ ]:
import numpy as np

# Define global parameters for consistency and easy modification
CELL_SIZE = 8 # in microns (for effective cell calculations)
DIAMETER_MIN_FILTER = None # Set to a number (e.g., 55) to filter out particles below this diameter, or None to disable

# Histogram and PDF Plotting Parameters
EFFECTIVE_CELL_HISTOGRAM_BINS = np.linspace(0, 600, 50) # Bins for Figure 6 effective cell calculations
HISTOGRAM_DENSITY_BINS = np.linspace(0, 600, 50) # Bins for density histograms (e.g., Figure 3)
HISTOGRAM_Y_LIMIT = 40 # Y-axis limit for percentage of effective cells plots (Figure 6)
HISTOGRAM_DENSITY_YLIM = (0, 0.03) # Y-axis limit for density histograms (e.g., Figure 3)
DIAMETER_BOXPLOT_YLIM = (0, 600) # Y-axis limit for Diameter box plots
CIRCULARITY_BOXPLOT_YLIM = (0.5, 1) # Y-axis limit for Circularity box plots
PDF_EVALUATION_RANGE = np.linspace(0, 600, 100) # X-axis range for PDF calculation, further extended to negative for potential KDE tails
PDF_PLOT_XLIM = (0, 700) # X-axis limit for PDF plots

# Statistical Test Parameters
KS_TEST_DAY1 = 'DAY2' # Initial day for KS test (corrected to uppercase 'DAY')
KS_TEST_DAY2 = 'DAY5' # Final day for KS test (corrected to uppercase 'DAY')

#Compound comperison

##Plots

### single batch

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Calculate mean and standard deviation of Diameter per DAY and Condition
summary_diameter = df.groupby(['DAY', 'Condition'], observed=False)['Diameter'].agg(['mean', 'std']).reset_index()
summary_diameter = summary_diameter.rename(columns={'mean': 'Mean Diameter', 'std': 'Std Dev Diameter'})

# Round the relevant columns to 2 decimal places
summary_diameter['Mean Diameter'] = summary_diameter['Mean Diameter'].round(2)
summary_diameter['Std Dev Diameter'] = summary_diameter['Std Dev Diameter'].round(2)

print("Summary Table: Mean and Standard Deviation of Diameter per Day and Condition")
display(summary_diameter)

# Create a FacetGrid to manage the subplot layout
g = sns.FacetGrid(
    data=df,
    # Removed: row='BATCH',     # Arrange batches vertically (as requested)
    col='DAY',       # Arrange days horizontally (as requested)
    col_wrap=5,      # Arrange days in a 2x2 grid
    hue='Condition', # Set hue to x-variable for coloring as requested
    height=6,   # Height of each facet
    aspect=0.5,  # Aspect ratio of each facet
    palette='viridis', # Apply palette for hue
)

# Map sns.boxplot to each facet.
# Pass x, y, and any specific boxprops directly to sns.boxplot.
g.map_dataframe(sns.boxplot, 'Condition', 'Diameter', boxprops={'edgecolor': 'black', 'linewidth': 1}, width=0.6)

# Set common properties for all facets
g.set_axis_labels('Condition', 'Diameter [um]') # Updated x-axis label
g.set_titles(col_template='{col_name}') # Title for each facet, adjusted since no 'row'
g.set(ylim=DIAMETER_BOXPLOT_YLIM) # Apply y-limit to all facets

# Add grid to each subplot and remove individual subplot legends
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75);
    ax.axhline(400, color='red', linestyle='--', linewidth=1.5) # Add red horizontal line at 400 diameter
    ax.axhline(200, color='orange', linestyle='--', linewidth=1.5) # Add orange horizontal line at 200 diameter
    ax.tick_params(axis='x', labelrotation=45) # Rotate x-axis labels by 45 degrees
    if ax.legend_: # Check if a legend exists before trying to remove it
        ax.legend_.remove() # Remove legend as hue is redundant with x-axis labels

# Add a main title to the entire figure, positioning its bottom slightly below the figure top
g.fig.suptitle('Diameter Distribution by Day and Condition', y=0.98) # Adjusted title

# Remove the overall legend as 'Condition' on x-axis makes hue legend redundant.
# plt.tight_layout adjusts subplot params for a tight layout.
# rect argument specifies the bounding box in (left, bottom, right, top) normalized coordinates that the subplots will fill.
# This leaves space for the suptitle.
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjusted rect for suptitle
plt.show()

### Circularity singular

### Multiple batches

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a FacetGrid to manage the subplot layout
g = sns.FacetGrid(
    data=df,
    row='BATCH',     # Arrange batches vertically (as requested)
    col='DAY',       # Arrange days horizontally (as requested)
    hue='Condition', # Set hue to x-variable for coloring as requested
    height=6,   # Height of each facet
    aspect=1.2,  # Aspect ratio of each facet
    palette='viridis', # Apply palette for hue
)

# Map sns.boxplot to each facet.
# Pass x, y, and any specific boxprops directly to sns.boxplot.
g.map_dataframe(sns.boxplot, 'Condition', 'Diameter', boxprops={'edgecolor': 'black', 'linewidth': 1}, width=0.6)

# Set common properties for all facets
g.set_axis_labels('Condition', 'Diameter [um]') # Updated x-axis label
g.set_titles(col_template='{col_name}', row_template='{row_name}') # Title for each facet
g.set(ylim=DIAMETER_BOXPLOT_YLIM) # Apply y-limit to all facets

# Add grid to each subplot and remove individual subplot legends
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75);
    if ax.legend_: # Check if a legend exists before trying to remove it
        ax.legend_.remove() # Remove legend as hue is redundant with x-axis labels

# Add a main title to the entire figure, positioning its bottom slightly below the figure top
g.fig.suptitle('Diameter Distribution by Batch, Day, and Condition', y=0.98) # Adjusted title

# Remove the overall legend as 'Condition' on x-axis makes hue legend redundant.
# plt.tight_layout adjusts subplot params for a tight layout.
# rect argument specifies the bounding box in (left, bottom, right, top) normalized coordinates that the subplots will fill.
# This leaves space for the suptitle.
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjusted rect for suptitle
plt.show()

### Circularity multi

In [ ]:
#FIGURE 2 Circularity boxplot
import matplotlib.pyplot as plt
import seaborn as sns

# Create a FacetGrid to manage the subplot layout for Circularity
g = sns.FacetGrid(
    data=df,
    row='BATCH',     # Arrange batches vertically (like Figure 1)
    col='DAY',       # Arrange days horizontally (like Figure 1)
    hue='Condition', # Set hue to x-variable for coloring
    height=6,
    aspect=1.2,
    palette='viridis', # Apply palette for hue
)

# Map sns.boxplot to each facet for Circularity.
# Pass x, y, and any specific boxprops directly to sns.boxplot.
g.map_dataframe(sns.boxplot, 'Condition', 'Circ.', boxprops={'edgecolor': 'black', 'linewidth': 0.5}, width=0.75) # Added width=0.75

# Set common properties for all facets
g.set_axis_labels('Condition', 'Circularity') # Updated x-axis label
g.set_titles(col_template='{col_name}', row_template='{row_name}') # Updated title templates
g.set(ylim=CIRCULARITY_BOXPLOT_YLIM) # Apply Circularity y-limit

# Add grid to each subplot and remove individual subplot legends
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75);
    if ax.legend_: # Check if a legend exists before trying to remove it
        ax.legend_.remove() # Remove legend as hue is redundant with x-axis labels

# Add a main title to the entire figure
g.fig.suptitle('ImageJ: Circularity Distribution by Batch, Day, and Condition', y=0.98) # Updated title

# Adjust layout to make space for the title
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjusted rect as legend is removed
plt.show()

In [ ]:
#FIGURE 2B Circularity boxplot
import matplotlib.pyplot as plt
import seaborn as sns

# Create a FacetGrid to manage the subplot layout for Circularity
g = sns.FacetGrid(
    data=df,
    # Removed: row='BATCH',     # Arrange batches vertically (like Figure 1)
    col='DAY',       # Arrange days horizontally (like Figure 1)
    col_wrap=2,      # Arrange days in a 2x2 grid
    hue='Condition', # Set hue to x-variable for coloring
    height=6,
    aspect=1.2,
    palette='viridis', # Apply palette for hue
)

# Map sns.boxplot to each facet for Circularity.
# Pass x, y, and any specific boxprops directly to sns.boxplot.
g.map_dataframe(sns.boxplot, 'Condition', 'Circ.', boxprops={'edgecolor': 'black', 'linewidth': 0.5}, width=0.75) # Added width=0.75

# Set common properties for all facets
g.set_axis_labels('Condition', 'Circularity') # Updated x-axis label
g.set_titles(col_template='Day: {col_name}') # Title for each facet, adjusted since no 'row'
g.set(ylim=CIRCULARITY_BOXPLOT_YLIM) # Apply Circularity y-limit

# Add grid to each subplot and remove individual subplot legends
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75);
    if ax.legend_: # Check if a legend exists before trying to remove it
        ax.legend_.remove() # Remove legend as hue is redundant with x-axis labels

# Add a main title to the entire figure
g.fig.suptitle('Circularity Distribution by Day and Condition', y=0.98) # Adjusted title

# Adjust layout to make space for the title
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjusted rect as legend is removed
plt.show()

### 200/200-400/400 plot

In [ ]:
import pandas as pd

# Define the diameter threshold
DIAMETER_THRESHOLD = 400

# Filter the DataFrame to include only data points where Diameter is below the threshold
df_below_threshold = df[df['Diameter'] < DIAMETER_THRESHOLD].copy()

# Group by 'DAY' and 'Condition' and count the number of data points (rows) in each group
# The .size() method counts rows per group
data_points_below_400 = df_below_threshold.groupby(['DAY', 'Condition'], observed=False).size().reset_index(name='Count_Below_400_Diameter')

# Display the results
print(f"Number of data points below {DIAMETER_THRESHOLD} Diameter per Day and Condition:")
display(data_points_below_400)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Define the diameter thresholds
DIAMETER_THRESHOLD_SMALL = 200
DIAMETER_THRESHOLD_LARGE = 400

# 1. Calculate total number of data points per DAY and Condition
total_counts = df.groupby(['DAY', 'Condition'], observed=False).size().reset_index(name='Total_Count')

# 2. Calculate data points below DIAMETER_THRESHOLD_SMALL (200)
df_below_200 = df[df['Diameter'] < DIAMETER_THRESHOLD_SMALL]
counts_below_200 = df_below_200.groupby(['DAY', 'Condition'], observed=False).size().reset_index(name='Count_Below_200_Diameter')

# 3. Calculate data points between DIAMETER_THRESHOLD_SMALL (200) and DIAMETER_THRESHOLD_LARGE (400)
df_between_200_and_400 = df[(df['Diameter'] >= DIAMETER_THRESHOLD_SMALL) & (df['Diameter'] < DIAMETER_THRESHOLD_LARGE)]
counts_between_200_and_400 = df_between_200_and_400.groupby(['DAY', 'Condition'], observed=False).size().reset_index(name='Count_Between_200_and_400_Diameter')

# 4. Calculate data points above DIAMETER_THRESHOLD_LARGE (400)
df_above_400 = df[df['Diameter'] >= DIAMETER_THRESHOLD_LARGE] # Changed to >= to include 400 in the upper bound category
counts_above_400 = df_above_400.groupby(['DAY', 'Condition'], observed=False).size().reset_index(name='Count_Above_400_Diameter')

# Merge counts into a single DataFrame
merged_counts = pd.merge(total_counts, counts_below_200, on=['DAY', 'Condition'], how='left')
merged_counts = pd.merge(merged_counts, counts_between_200_and_400, on=['DAY', 'Condition'], how='left')
merged_counts = pd.merge(merged_counts, counts_above_400, on=['DAY', 'Condition'], how='left')

# Fill NaN values (where no particles were found in a category) with 0
merged_counts['Count_Below_200_Diameter'] = merged_counts['Count_Below_200_Diameter'].fillna(0).astype(int)
merged_counts['Count_Between_200_and_400_Diameter'] = merged_counts['Count_Between_200_and_400_Diameter'].fillna(0).astype(int)
merged_counts['Count_Above_400_Diameter'] = merged_counts['Count_Above_400_Diameter'].fillna(0).astype(int)

# 5. Calculate percentages
merged_counts['Percentage_Below_200'] = (merged_counts['Count_Below_200_Diameter'] / merged_counts['Total_Count']) * 100
merged_counts['Percentage_Between_200_and_400'] = (merged_counts['Count_Between_200_and_400_Diameter'] / merged_counts['Total_Count']) * 100
merged_counts['Percentage_Above_400'] = (merged_counts['Count_Above_400_Diameter'] / merged_counts['Total_Count']) * 100

# 6. Prepare data for plotting (melt to long format)
plot_df = merged_counts.melt(
    id_vars=['DAY', 'Condition'],
    value_vars=['Percentage_Below_200', 'Percentage_Between_200_and_400', 'Percentage_Above_400'],
    var_name='Threshold_Type',
    value_name='Percentage'
)

# Rename threshold types for better plot labels
plot_df['Threshold_Type'] = plot_df['Threshold_Type'].replace({
    'Percentage_Below_200': f'Below {DIAMETER_THRESHOLD_SMALL} µm',
    'Percentage_Between_200_and_400': f'{DIAMETER_THRESHOLD_SMALL}-{DIAMETER_THRESHOLD_LARGE} µm',
    'Percentage_Above_400': f'Above {DIAMETER_THRESHOLD_LARGE} µm'
})

# Define custom palette for the new categories
custom_palette = {
    f'Below {DIAMETER_THRESHOLD_SMALL} µm': 'green',
    f'{DIAMETER_THRESHOLD_SMALL}-{DIAMETER_THRESHOLD_LARGE} µm': 'orange',
    f'Above {DIAMETER_THRESHOLD_LARGE} µm': 'red'
}

# Ensure 'Condition' and 'DAY' are categorical for correct ordering
plot_df['Condition'] = pd.Categorical(plot_df['Condition'], categories=CONDITION_ORDER, ordered=True)
plot_df['DAY'] = pd.Categorical(plot_df['DAY'], categories=df['DAY'].cat.categories, ordered=True)

# 7. Generate the grouped bar plot
g = sns.catplot(
    data=plot_df,
    x='Condition',
    y='Percentage',
    hue='Threshold_Type',
    col='DAY',
    col_wrap=2, # Wrap columns for better layout if many days
    kind='bar',
    palette=custom_palette, # Use custom palette
    height=5,
    aspect=1.2,
    errorbar=None, # No error bars for these calculated percentages
    width=0.8,
    legend_out=True # Ensure the legend is placed outside the subplots
)

g.set_axis_labels('Condition', 'Percentage of Particles (%)')
g.set_titles(col_template='{col_name}')
g.fig.suptitle(
    f'Percentage of Particles by Diameter Threshold per Day and Condition',
    y=1.03
)

# Add grid and rotate x-axis labels
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(0, 100) # Percentage goes from 0 to 100

# Customize the legend created by catplot
g.legend.set_title('Threshold Type')
g.legend.set_loc('upper right')
g.legend.set_bbox_to_anchor((1.02, 1.05))
g.legend.set_frame_on(False)
g.legend.set_ncols(3) # Correct for three categories

plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout to make space for the title and legend
plt.show()

# 8. Add summary table for percentages by diameter thresholds
print("\n--- Summary: Percentage of Particles by Diameter Thresholds ---")
summary_percentages_df = merged_counts[['DAY', 'Condition', 'Percentage_Below_200', 'Percentage_Between_200_and_400', 'Percentage_Above_400']].copy()
# Round percentages for display
summary_percentages_df['Percentage_Below_200'] = summary_percentages_df['Percentage_Below_200'].round(2)
summary_percentages_df['Percentage_Between_200_and_400'] = summary_percentages_df['Percentage_Between_200_and_400'].round(2)
summary_percentages_df['Percentage_Above_400'] = summary_percentages_df['Percentage_Above_400'].round(2)

display(summary_percentages_df)

### Compounds vs NEG diameter reduction

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Calculate condition means using pivot_table
means = df.pivot_table(
    index=['BATCH', 'DAY'],
    columns='Condition',
    values='Diameter',
    aggfunc='mean',
    observed=False,
)

# 2. Vectorized calculation of percentage reduction relative to NEG
pct_reduction = (
    means.apply(lambda col: ((means['NEG'] - col) / means['NEG']) * 100)
    .drop(columns=['NEG'])
    .reset_index()
)

# 3. Reshape (melt) data for plotting
plot_df = pct_reduction.melt(
    id_vars=['BATCH', 'DAY'],
    var_name='Condition',
    value_name='Percentage_Reduction_vs_NEG',
)

# 4. Filter for DAY5 and enforce categorical ordering
CONDITION_ORDER = [
    'DS',
    'HS',
    'PEG',
    'PVA',
    'DSPEG',
    'DSPVA',
    'HSPEG',
]  # Excludes NEG
day5_df = plot_df[plot_df['DAY'] == 'DAY5'].copy()
day5_df['Condition'] = pd.Categorical(
    day5_df['Condition'], categories=CONDITION_ORDER, ordered=True
)

# 5. Plotting
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=day5_df,
    x='Condition',
    y='Percentage_Reduction_vs_NEG',
    hue='Condition',
    palette='viridis',
    width=0.75,
)

ax.set_title('Diameter Percentage Reduction vs. NEG (Day 5)', pad=15)
ax.set_xlabel('Condition')
ax.set_ylabel('Diameter Reduction vs NEG (%)')
ax.grid(True, linestyle='--', alpha=0.75)
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Histograms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re

# Define the target batch for this analysis (consistent with statistical analysis)
# TARGET_BATCH_VIS = 'B5.1P3R6'

# Filter the DataFrame to include only the target batch
# df_filtered = df[df['BATCH'] == TARGET_BATCH_VIS].copy()
df_filtered = df.copy()

# Ensure 'Condition' is a categorical type for proper sorting and unique value handling
# First, get unique conditions and sort them naturally if needed, similar to how 'DAY' is handled
# Assuming 'Condition' might have a natural order or you want to maintain an order.
# For now, we'll just convert it to categorical without specific order if not explicitly defined yet.
# If there's a specific order needed for 'Condition', we should define 'sorted_conditions' similar to 'sorted_days'.
# For the purpose of fixing the immediate error, we'll just convert it.

# Get unique conditions present in the data from the filtered DataFrame
# This step is still useful to identify which conditions are actually present in the data
actual_conditions_in_df = [cond for cond in df_filtered['Condition'].unique() if pd.notna(cond)]

# Filter CONDITION_ORDER to only include conditions actually present in the DataFrame
# This maintains the desired order while excluding conditions not in the current data
ordered_conditions_for_plot = [cond for cond in CONDITION_ORDER if cond in actual_conditions_in_df]

df_filtered['Condition'] = pd.Categorical(df_filtered['Condition'], categories=ordered_conditions_for_plot, ordered=True)

unique_conditions = df_filtered['Condition'].cat.categories # Now this will work

# Explicitly set the DAY column as categorical with day_order for consistent plotting
df_filtered['DAY'] = pd.Categorical(df_filtered['DAY'], categories=day_order, ordered=True)
unique_days = df_filtered['DAY'].cat.categories

n_conditions = len(unique_conditions)

# Determine grid size for subplots based on conditions
n_cols = 4 # Changed from 2 to 3
n_rows = (n_conditions + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 7 * 0.75, n_rows * 6 * 0.75))
axes = axes.flatten()
bins = HISTOGRAM_DENSITY_BINS # Use configured bins

min_diameter = df_filtered['Diameter'].min()
max_diameter = df_filtered['Diameter'].max()

for i, condition in enumerate(unique_conditions):
    ax = axes[i]
    # Subset the DataFrame for the current condition using the filtered DataFrame
    condition_df = df_filtered[df_filtered['Condition'] == condition]

    # Plot histogram for the current condition, with 'DAY' as hue
    sns.histplot(data=condition_df, bins=bins, x='Diameter', hue='DAY', stat='percent', common_norm=False, kde=True, ax=ax, palette=color_map, multiple="layer")
    ax.set_title(f'{condition} Particle Diameter Distribution')
    ax.set_xlabel('Diameter')
    ax.set_ylabel('Population Percentage (%)')
    ax.grid(axis='y', alpha=0.75)
    ax.set_ylim(0, 35) # Set constant y-axis limit as requested
    ax.set_xlim(0, bins.max()) # Ensure x-axis starts at 0

    # Add vertical reference lines
    ax.axvline(200, color='orange', linestyle='--', linewidth=1.5)
    ax.axvline(400, color='red', linestyle='--', linewidth=1.5)

    # Get handles and labels from the current axes after plotting.
    # This will return what histplot actually created for the legend.
    handles, labels = ax.get_legend_handles_labels()

    # Only show legend if there are actual handles (artists) to display.
    # This prevents the UserWarning when no legend items are generated.
    if handles:
        # histplot automatically creates a legend if hue is used. We get its handles and labels
        # and then explicitly set the legend to avoid potential duplicate legends or issues.
        ax.legend(handles=handles, labels=labels, title='DAY')

# Hide any unused subplots
for j in range(n_conditions, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import re
from scipy.stats import gaussian_kde

# Removed: Target batch filtering as requested.
# ---------------------------------------------------------
# Target batch
# ---------------------------------------------------------
# TARGET_BATCH_VIS = 'B5.1P3R6'

# Use the entire DataFrame as filtering is removed
df_filtered = df.copy()

# ---------------------------------------------------------
# Sort conditions naturally
# ---------------------------------------------------------
def natural_sort_key(s):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r'([0-9]+)', str(s))
    ]

unique_conditions_list = (
    df_filtered['Condition']
    .dropna()
    .unique()
)

sorted_conditions = sorted(
    unique_conditions_list,
    key=natural_sort_key
)

# Keep condition ordering
df_filtered['Condition'] = pd.Categorical(
    df_filtered['Condition'],
    categories=sorted_conditions,
    ordered=True
)

unique_conditions = df_filtered['Condition'].cat.categories

# ---------------------------------------------------------
# Day ordering
# ---------------------------------------------------------
unique_days = list(df_filtered['DAY'].dropna().unique())

# Optional: explicitly define order (kept for consistent plotting of days if needed)
day_order = ['DAY1', 'DAY2', 'DAY3', 'DAY4', 'DAY5']
unique_days = [d for d in day_order if d in unique_days]

# ---------------------------------------------------------
# Cell size and cell volume
# ---------------------------------------------------------
cell_size = 8  # µm

cell_volume = (4 / 3) * np.pi * (cell_size / 2)**3

# ---------------------------------------------------------
# Diameter evaluation grid
# ---------------------------------------------------------
x_min = 0
x_max = 600

x_evaluation = np.linspace(x_min, x_max, 1000)

# ---------------------------------------------------------
# Colors
# ---------------------------------------------------------
color_map = {
    "DAY1": "#000080", # Dark blue for DAY1
    "DAY2": "#414487",
    "DAY3": "#2A788E",
    "DAY4": "#22A884",
    "DAY5": "#7AD151",
}


# ---------------------------------------------------------
# Figure layout
# ---------------------------------------------------------
n_conditions = len(unique_conditions)

n_cols = 4
n_rows = int(np.ceil(n_conditions / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(n_cols * 5.4, n_rows * 4.2),
    sharex=True,
    sharey=True
)

axes = np.atleast_1d(axes).flatten()

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
for i, condition in enumerate(unique_conditions):

    ax = axes[i]

    condition_df = df_filtered[
        df_filtered['Condition'] == condition
    ]

    for day in unique_days:

        day_data = (
            condition_df[
                condition_df['DAY'] == day
            ]['Diameter']
            .dropna()
            .values
        )

        # Need at least 2 observations
        if len(day_data) < 2:
            continue

        # -------------------------------------------------
        # Cell-weighting
        #
        # Number of cells in an aggregate is assumed
        # proportional to aggregate volume.
        # -------------------------------------------------
        aggregate_volume = (
            (4 / 3) * np.pi * (day_data / 2)**3
        )

        weights = aggregate_volume / cell_volume

        # -------------------------------------------------
        # Weighted KDE
        # -------------------------------------------------
        kde = gaussian_kde(
            day_data,
            weights=weights
        )

        y_pdf = kde(x_evaluation)

        # Convert probability density to %
        y_pdf_percent = y_pdf * 100

        ax.plot(
            x_evaluation,
            y_pdf_percent,
            label=day,
            color=color_map.get(day, None),
            linewidth=2.5
        )

    # Add vertical lines at 200 and 400 diameter
    ax.axvline(200, color='orange', linestyle='--', linewidth=1.5)
    ax.axvline(400, color='red', linestyle='--', linewidth=1.5)

    # -----------------------------------------------------
    # Panel title
    # -----------------------------------------------------
    ax.set_title(
        f'{condition}',
        fontsize=11
    )

    # -----------------------------------------------------
    # Axes
    # -----------------------------------------------------
    ax.set_xlabel('Aggregate Diameter (µm)', fontsize=10)
    ax.set_ylabel(
        'Probability Density (% µm⁻¹)',
        fontsize=10
    )

    ax.set_xlim(0, 600)

    # Adjust depending on your data
    ax.set_ylim(0, 2.0)

    # -----------------------------------------------------
    # Grid
    # -----------------------------------------------------
    ax.grid(
        True,
        linestyle='--',
        alpha=0.35
    )

    # -----------------------------------------------------
    # Legend
    # -----------------------------------------------------
    ax.legend(
        title='Culture Day',
        loc='upper right',
        frameon=True
    )

# ---------------------------------------------------------
# Remove unused axes
# ---------------------------------------------------------
for j in range(n_conditions, len(axes)):
    fig.delaxes(axes[j])

# ---------------------------------------------------------
# Overall layout
# ---------------------------------------------------------
fig.suptitle(
    'Cell-Weighted Probability Distribution of Aggregate Diameter',
    fontsize=15,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.integrate import trapezoid
import numpy as np
from scipy.stats import gaussian_kde # Ensure gaussian_kde is imported as it's used

print("--- Validation: Area Under Curve (AUC) for Figure 7 PDF ---")

# Define unique_plates from the DataFrame
unique_plates = df['PLATE'].unique()

for plate_type in unique_plates:
    df_plate = df[df['PLATE'] == plate_type]
    print(f"\nPlate: {plate_type}")

    for day in unique_days:
        day_data = df_plate[df_plate['DAY'] == day]['Diameter'].dropna().values

        if len(day_data) > 1:
            # Check data range for F125 Day1 if this is the target
            if plate_type == 'F125' and day == 'Day1':
                print(f"  {day} Diameter range: min={day_data.min():.2f}, max={day_data.max():.2f}")
                print(f"  PDF Evaluation Range (x_evaluation): min={x_evaluation.min():.2f}, max={x_evaluation.max():.2f}")

            # Re-calculate the KDE to get the Y values
            weights = (4/3) * np.pi * (day_data/2)**3 / CELL_SIZE
            kde = gaussian_kde(day_data, weights=weights)
            y_pdf = kde(x_evaluation)

            # Calculate AUC using trapezoidal rule
            # We use the original y_pdf (not multiplied by 100) for mathematical validation
            auc = trapezoid(y_pdf, x_evaluation)
            print(f"  {day}: AUC = {auc:.4f}")

## Statistics between compounds

### Uniformity and lowest mean diameter ranking

In [ ]:
from scipy.stats import ks_2samp
import pandas as pd
import itertools

# Define the target batch for this analysis (commented out as per user request)
# TARGET_BATCH = 'STD-MAN-R5'

# Filter the main DataFrame and summary DataFrame for the target batch
# Changed to use the entire dataframe as batch filtering is removed
df_for_analysis = df.copy()
summary_df_for_analysis = summary_df.copy()

# Check if the filtered data is empty, without specific batch reference
if df_for_analysis.empty:
    print(f"Warning: No data available. All statistical tests in this cell will be skipped or may produce NaNs.")

# --- Original KS Test: Day 1 vs Day 3 ---
# Updated print statement to remove batch reference
print(f"--- Kolmogorov-Smirnov Test: {KS_TEST_DAY1} vs {KS_TEST_DAY2} (All Batches) ---")
# Extract diameter data for Day 1 and Day 3 from the filtered DataFrame
day1_data = df_for_analysis[df_for_analysis['DAY'] == KS_TEST_DAY1]['Diameter'].dropna()
day3_data = df_for_analysis[df_for_analysis['DAY'] == KS_TEST_DAY2]['Diameter'].dropna()

if len(day1_data) > 1 and len(day3_data) > 1: # Ensure enough data points for the test
    # Perform the two-sample Kolmogorov-Smirnov test
    statistic, p_value = ks_2samp(day1_data, day3_data)

    print(f"KS Statistic: {statistic:.4f}")
    print(f"P-value: {p_value:.2e}")

    if p_value < 0.05:
        print("\nResult: The difference in distributions is statistically significant (p < 0.05).")
    else:
        print("\nResult: The difference in distributions is NOT statistically significant (p >= 0.05).")
elif len(day1_data) == 0:
    print(f"Warning: No data available for {KS_TEST_DAY1}. KS test cannot be performed.")
elif len(day3_data) == 0:
    print(f"Warning: No data available for {KS_TEST_DAY2}. KS test cannot be performed.")
else:
    print(f"Warning: Not enough data points for {KS_TEST_DAY1} or {KS_TEST_DAY2} to perform KS test.")
print("-" * 50)

# --- New Section: Pairwise KS Tests between Conditions on the Same Day ---
# Updated print statement to remove batch reference
print(f"\n--- Pairwise Kolmogorov-Smirnov Tests: Conditions on the Same Day (All Batches) ---")

unique_days = df['DAY'].cat.categories # Get sorted unique days from the original df to maintain global order

for day in unique_days:
    print(f"\nDay: {day}")
    # Get conditions present for the current day
    conditions_on_day = df_for_analysis[df_for_analysis['DAY'] == day]['Condition'].unique().tolist()
    conditions_on_day.sort() # Sort for consistent output

    if len(conditions_on_day) < 2:
        print(f"  Not enough conditions ({len(conditions_on_day)}) for pairwise comparison on {day}.")
        continue

    for cond1, cond2 in itertools.combinations(conditions_on_day, 2):
        data_cond1 = df_for_analysis[(df_for_analysis['DAY'] == day) & (df_for_analysis['Condition'] == cond1)]['Diameter'].dropna()
        data_cond2 = df_for_analysis[(df_for_analysis['DAY'] == day) & (df_for_analysis['Condition'] == cond2)]['Diameter'].dropna()

        if len(data_cond1) > 1 and len(data_cond2) > 1: # Ensure enough data points for the test
            statistic_pair, p_value_pair = ks_2samp(data_cond1, data_cond2)
            print(f"  {cond1} vs {cond2}: KS Stat = {statistic_pair:.4f}, P-value = {p_value_pair:.2e}", end="")
            if p_value_pair < 0.05:
                print(" (Significant)")
            else:
                print(" (Not significant)")
        else:
            print(f"  Not enough data for {cond1} or {cond2} on {day} for KS test.")
print("-" * 50)

# --- New Section: Identify Most Uniform Condition per Day ---
# Updated print statement to remove batch reference
print(f"\n--- Most Uniform Condition (Least Diameter Standard Deviation) per Day (All Batches) ---")

# summary_df_for_analysis is used here
for day in unique_days:
    day_summary = summary_df_for_analysis[summary_df_for_analysis['DAY'] == day]

    if not day_summary.empty:
        # Find the condition with the minimum standard deviation for Diameter
        most_uniform_condition_row = day_summary.loc[day_summary['Dia_std'].idxmin()]
        condition_name = most_uniform_condition_row['Condition']
        std_dev = most_uniform_condition_row['Dia_std']
        mean_dia = most_uniform_condition_row['Dia_mean'] # Get the mean diameter
        print(f"   {day}: Most uniform condition is '{condition_name}' with Dia_mean = {mean_dia:.2f}, Dia_std = {std_dev:.2f}")
    else:
        print(f"   {day}: No data available to determine most uniform condition.")
print("-" * 50)

# --- New Section: Identify Lowest Mean Diameter Condition per Day ---
# Updated print statement to remove batch reference
print(f"\n--- Condition with Lowest Mean Diameter per Day (All Batches) ---")

for day in unique_days:
    day_summary = summary_df_for_analysis[summary_df_for_analysis['DAY'] == day]

    if not day_summary.empty:
        # Find the condition with the minimum mean diameter
        lowest_mean_condition_row = day_summary.loc[day_summary['Dia_mean'].idxmin()]
        condition_name = lowest_mean_condition_row['Condition']
        mean_dia = lowest_mean_condition_row['Dia_mean']
        std_dev = lowest_mean_condition_row['Dia_std'] # Also display std for context
        print(f"  {day}: Lowest mean diameter is '{condition_name}' with Dia_mean = {mean_dia:.2f}, Dia_std = {std_dev:.2f}")
    else:
        print(f"  {day}: No data available to determine lowest mean diameter.")
print("-" * 50)

# --- New Section: Ranking of Uniformity per Day ---
# Updated print statement to remove batch reference
print(f"\n--- Ranking of Conditions (Most Uniform to Least Uniform) per Day (All Batches) ---")

for day in unique_days:
    day_summary = summary_df_for_analysis[summary_df_for_analysis['DAY'] == day]

    if not day_summary.empty:
        # Sort by Dia_std to rank from most uniform (lowest std) to least uniform (highest std)
        ranked_conditions = day_summary.sort_values(by='Dia_std', ascending=True)
        print(f"\n  {day} Ranking:")
        for rank, (index, row) in enumerate(ranked_conditions.iterrows()):
            print(f"    {rank + 1}. {row['Condition']}  Dia_mean = {row['Dia_mean']:.2f}, (Dia_std = {row['Dia_std']:.2f} )")
    else:
        print(f"\n  {day}: No data available to rank conditions.")
print("-" * 50)

## Kruskal-Wallis+ Holm corrected

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from scipy.stats import chi2

print("--- Mixed-effects model: Diameter ~ Condition + DAY + (1|Well), DAY2-5 only ---")

CONDITION_ORDER = ['DS', 'HS', 'PEG', 'PVA', 'DSPEG', 'DSPVA', 'HSPEG', 'NEG']

# df must already contain 'Diameter' (2*sqrt(Area/pi)) and 'Condition'/'DAY' columns.

# ---- Restrict to DAY2-DAY5 (exclude DAY1, which was sourced from ZEI) ----
df = df[df['DAY'].isin(['DAY2', 'DAY3', 'DAY4', 'DAY5'])].copy()

# NEG as reference -> coefficients = compound vs NEG
df['Condition'] = pd.Categorical(
    df['Condition'],
    categories=['NEG'] + [c for c in CONDITION_ORDER if c != 'NEG'],
    ordered=True)
df['DAY']  = pd.Categorical(df['DAY'], categories=['DAY2','DAY3','DAY4','DAY5'], ordered=True)
df['Well'] = df['DAY'].astype(str) + '_' + df['Condition'].astype(str)
df['Dia100'] = df['Diameter'] / 100.0   # scale for numerical stability

def holm_adjust(pvals):
    p = np.asarray(pvals, float); n = len(p)
    o = np.argsort(p); adj = np.empty(n); prev = 0.0
    for r, i in enumerate(o):
        adj[i] = min(max((n - r) * p[i], prev), 1.0); prev = adj[i]
    return adj

# ICC (one-way random effects) — degree of within-well clustering
g = df.groupby('Well')['Diameter']
k = g.count().mean()
msb = np.sum(g.count().values * (g.mean().values - df['Diameter'].mean())**2) / (g.ngroups - 1)
msw = np.sum([(df.loc[df['Well']==w, 'Diameter'].var(ddof=1) * (len(df.loc[df['Well']==w]) - 1))
              for w in g.groups]) / (len(df) - g.ngroups)
icc = (msb - msw) / (msb + (k - 1) * msw)
print(f"wells={g.ngroups}  aggregates={len(df)}  ICC={icc:.3f}  (clustering {'high' if icc>0.1 else 'low'})")

# Additive mixed model (interaction is confounded with Well: 1 well per Condition x DAY)
m  = smf.mixedlm("Dia100 ~ Condition + DAY", df, groups="Well").fit(method='bfgs', maxiter=2000)
m0 = smf.mixedlm("Dia100 ~ DAY",            df, groups="Well").fit(method='bfgs', maxiter=2000)

lr = 2 * (m.llf - m0.llf); dof = len(m.params) - len(m0.params)
print(f"\nCondition omnibus LRT: chi2={lr:.1f} df={dof} p={chi2.sf(lr, dof):.3e}")

terms = [t for t in m.params.index if t.startswith('Condition[')]
pv = [m.pvalues[t] for t in terms]; adj = holm_adjust(pv)
print("\nCompound vs NEG (Holm-adjusted p-values):")
print(f"{'compound':10s}{'diff(um)':>9s}{'p_raw':>11s}{'p_holm':>9s}")
for t, pa in zip(terms, adj):
    c = t.split('[')[1].rstrip(']')
    flag = '  *' if pa < 0.05 else ''
    print(f"{c:10s}{m.params[t]*100:9.2f}{m.pvalues[t]:11.3e}{pa:9.3f}{flag}")

###ANOVA and Tukey's HSD Post-Hoc Tests

In [ ]:
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import re # Needed for natural_sort_key if used for batch sorting

print("--- ANOVA and Tukey's HSD Post-Hoc Tests by Day and Metric ---")

# Helper function for natural sorting, typically defined elsewhere but included for robustness
def natural_sort_key(s):
    if pd.isna(s):
        return []
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'([0-9]+)', str(s))]

# Ensure 'DAY' column in summary_df is categorical
# Try to use 'sorted_days' from the kernel state, which was established for df['DAY'] in a previous cell.
# If 'sorted_days' is not available or not a list, fall back to natural sorting unique days from summary_df.
if 'sorted_days' in globals() and isinstance(globals()['sorted_days'], list):
    if not isinstance(summary_df['DAY'].dtype, pd.CategoricalDtype):
        summary_df['DAY'] = pd.Categorical(summary_df['DAY'], categories=sorted_days, ordered=True)
else:
    unique_days_from_summary = summary_df['DAY'].dropna().unique()
    sorted_unique_days = sorted(unique_days_from_summary, key=natural_sort_key)
    summary_df['DAY'] = pd.Categorical(summary_df['DAY'], categories=sorted_unique_days, ordered=True)

# Ensure 'BATCH' column in summary_df is categorical
if not isinstance(summary_df['BATCH'].dtype, pd.CategoricalDtype):
    # Get unique batches from df and sort them naturally to define categories
    unique_batches_from_df = df['BATCH'].dropna().unique()
    sorted_unique_batches = sorted(unique_batches_from_df, key=natural_sort_key)
    summary_df['BATCH'] = pd.Categorical(summary_df['BATCH'], categories=sorted_unique_batches, ordered=True)

# Ensure 'DAY' and 'BATCH' columns in the original 'df' are also categorical for filtering
if not isinstance(df['DAY'].dtype, pd.CategoricalDtype):
    if 'sorted_days' in globals() and isinstance(globals()['sorted_days'], list):
        df['DAY'] = pd.Categorical(df['DAY'], categories=sorted_days, ordered=True)
    else:
        unique_days_from_df = df['DAY'].dropna().unique()
        sorted_unique_days_df = sorted(unique_days_from_df, key=natural_sort_key)
        df['DAY'] = pd.Categorical(df['DAY'], categories=sorted_unique_days_df, ordered=True)

if not isinstance(df['BATCH'].dtype, pd.CategoricalDtype):
    unique_batches_from_df = df['BATCH'].dropna().unique()
    sorted_unique_batches_df = sorted(unique_batches_from_df, key=natural_sort_key)
    df['BATCH'] = pd.Categorical(df['BATCH'], categories=sorted_unique_batches_df, ordered=True)

unique_days = summary_df['DAY'].cat.categories
unique_batches = summary_df['BATCH'].cat.categories # Assuming multiple batches if 'BATCH' is in summary_df

for day in unique_days:
    print(f"\nAnalyzing Day: {day}")

    for batch in unique_batches:
        # Filter the ORIGINAL DataFrame 'df' to get raw data for the current day and batch
        data_for_analysis = df[(df['DAY'] == day) & (df['BATCH'] == batch)].copy()

        if not data_for_analysis.empty:
            print(f"  Batch: {batch}")
            unique_conditions = data_for_analysis['Condition'].dropna().unique()

            if len(unique_conditions) < 2:
                print(f"    Not enough unique conditions ({len(unique_conditions)}) to perform ANOVA for this Batch and Day.")
                continue

            # --- Diameter Analysis ---
            # Extract raw 'Diameter' values for each condition from the original data
            diameter_groups = [data_for_analysis[data_for_analysis['Condition'] == cond]['Diameter'].dropna().values for cond in unique_conditions]

            # Filter out groups with insufficient data (ANOVA requires at least 2 data points per group)
            diameter_groups = [group for group in diameter_groups if len(group) > 1]

            if len(diameter_groups) >= 2:
                f_stat_dia, p_val_dia = f_oneway(*diameter_groups)
                print(f"    Diameter - ANOVA F-statistic: {f_stat_dia:.2f}, P-value: {p_val_dia:.3e}")

                if p_val_dia < 0.05:
                    print("      Significant difference detected in Diameter means across conditions. Performing Tukey's HSD...")
                    # Prepare data for Tukey's HSD (requires at least 2 data points per group)
                    data_for_tukey_dia = []
                    labels_for_tukey_dia = []
                    for i, cond in enumerate(unique_conditions):
                        current_group = data_for_analysis[data_for_analysis['Condition'] == cond]['Diameter'].dropna().values
                        if len(current_group) > 1: # Only include groups with >1 data point for Tukey's
                            data_for_tukey_dia.extend(current_group)
                            labels_for_tukey_dia.extend([cond] * len(current_group))

                    if len(set(labels_for_tukey_dia)) > 1: # Need at least two distinct groups for Tukey's
                        tukey_result_dia = pairwise_tukeyhsd(endog=data_for_tukey_dia, groups=labels_for_tukey_dia, alpha=0.05)
                        print(tukey_result_dia.summary())
                    else:
                        print("      Not enough distinct groups (each with >1 data point) for Tukey's HSD on Diameter.")
                else:
                    print("      No significant difference in Diameter means across conditions.")
            else:
                print("    Not enough valid data groups (each with >1 data point) for Diameter ANOVA.")

            # --- Circularity Analysis ---
            # Extract raw 'Circ.' values for each condition from the original data
            circularity_groups = [data_for_analysis[data_for_analysis['Condition'] == cond]['Circ.'].dropna().values for cond in unique_conditions]

            # Filter out groups with insufficient data (ANOVA requires at least 2 data points per group)
            circularity_groups = [group for group in circularity_groups if len(group) > 1]

            if len(circularity_groups) >= 2:
                f_stat_circ, p_val_circ = f_oneway(*circularity_groups)
                print(f"    Circularity - ANOVA F-statistic: {f_stat_circ:.2f}, P-value: {p_val_circ:.3e}")

                if p_val_circ < 0.05:
                    print("      Significant difference detected in Circularity means across conditions. Performing Tukey's HSD...")
                    # Prepare data for Tukey's HSD (requires at least 2 data points per group)
                    data_for_tukey_circ = []
                    labels_for_tukey_circ = []
                    for i, cond in enumerate(unique_conditions):
                        current_group = data_for_analysis[data_for_analysis['Condition'] == cond]['Circ.'].dropna().values
                        if len(current_group) > 1: # Only include groups with >1 data point for Tukey's
                            data_for_tukey_circ.extend(current_group)
                            labels_for_tukey_circ.extend([cond] * len(current_group))

                    if len(set(labels_for_tukey_circ)) > 1: # Need at least two distinct groups for Tukey's
                        tukey_result_circ = pairwise_tukeyhsd(endog=data_for_tukey_circ, groups=labels_for_tukey_circ, alpha=0.05)
                        print(tukey_result_circ.summary())
                    else:
                        print("      Not enough distinct groups (each with >1 data point) for Tukey's HSD on Circularity.")
                else:
                    print("      No significant difference in Circularity means across conditions.")
            else:
                print("    Not enough valid data groups (each with >1 data point) for Circularity ANOVA.")

        else:
            print(f"  No data available in original 'df' for Batch: {batch} on Day: {day}")


#Method comparison

##Plots

### Diameter comperison plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches # For creating proxy artists

# Create a catplot to generate box plots comparing batches across all days
g = sns.catplot(
    data=df, # Use the full DataFrame to include all days
    x='Condition',   # Conditions on x-axis
    y='Diameter',
    hue='BATCH',     # Batches as hue for comparison within conditions
    hue_order=BATCH_ORDER, # Explicitly set the order of batches
    row='DAY',       # Arrange days vertically in separate rows
    kind='box',
    palette='viridis', # Apply palette for hue
    height=3,        # Adjusted height for multiple rows
    aspect=5.0,      # Adjusted aspect ratio for wider plots (was 2.5, now doubled)
    boxprops={'edgecolor': 'black', 'linewidth': 0.5}, # Consistent box properties
    width=0.75,
    legend=False     # Suppress the default FacetGrid legend
)

# Set common properties for all facets
g.set_axis_labels('Condition', 'Diameter [µm]')
g.set_titles(row_template='{row_name}') # Title for each row (Day)
g.set(ylim=DIAMETER_BOXPLOT_YLIM)

# Add grid to each subplot
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75)
    ax.tick_params(axis='x', rotation=45) # Rotate x-axis labels if they overlap

# Add a main title to the entire figure, positioned slightly higher
g.fig.suptitle('Diameter Distribution by Day, Condition, and Batch', y=1.03)

# Prepare proxy artists (patches) for both legends, as we're creating them manually.
# Use BATCH_ORDER directly to ensure consistent legend order
unique_batches_for_legend = BATCH_ORDER # Use the predefined order for legends
palette_colors = sns.color_palette('viridis', n_colors=len(unique_batches_for_legend))
batch_color_map = {batch: color for batch, color in zip(unique_batches_for_legend, palette_colors)}
proxy_handles = [mpatches.Patch(color=batch_color_map[batch], label=batch) for batch in unique_batches_for_legend]
labels = list(unique_batches_for_legend) # Labels for the legend

# 1. Create the horizontal legend under the title (figure-level).
unique_batches_count = len(unique_batches_for_legend)
legend_horizontal = g.fig.legend(handles=proxy_handles, labels=labels,
                                 loc='upper center', bbox_to_anchor=(0.5, 0.98), # Position near top-center
                                 ncol=unique_batches_count, title='BATCH', frameon=False,
                                 fontsize='small') # Unique property for separate legend instance

# 2. Create the second 'default' style legend in the top-right corner (figure-level).
legend_top_right = g.fig.legend(handles=proxy_handles, labels=labels,
                                loc='upper right', bbox_to_anchor=(0.99, 0.95), # Position near top-right of figure
                                title='BATCH', frameon=True,
                                fontsize='medium') # Unique property for separate legend instance

# Adjust layout to make space for the title and both figure-level legends.
# The rect argument defines the bounding box for the subplots.
# (left, bottom, right, top). We leave space at the top (0.9).
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

###Diameter comperison but only 1 day

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches # For creating proxy artists

# Filter the DataFrame to include only 'DAY5'
df_day5 = df[df['DAY'] == 'DAY5'].copy()

# Create a catplot to generate box plots comparing batches for DAY5
g = sns.catplot(
    data=df_day5,
    x='Condition',   # Conditions on x-axis
    y='Diameter',
    hue='BATCH',     # Batches as hue for comparison within conditions
    hue_order=BATCH_ORDER, # Explicitly set the order of batches
    kind='box',
    palette='viridis', # Apply palette for hue
    height=7,        # Height of the single facet
    aspect=3.0,      # Aspect ratio for a wider single plot
    boxprops={'edgecolor': 'black', 'linewidth': 0.5}, # Consistent box properties
    width=0.75,
    legend=False     # Suppress the default FacetGrid legend
)

# Set common properties for the single facet
g.set_axis_labels('Condition', 'Diameter [µm]')
g.set(ylim=DIAMETER_BOXPLOT_YLIM)

# Add grid to the subplot
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75)
    ax.tick_params(axis='x', rotation=45) # Rotate x-axis labels if they overlap

# Add a main title to the entire figure, positioned slightly higher
g.fig.suptitle('Diameter Distribution for Day 5 by Condition and Batch', y=1.03)

# Prepare proxy artists (patches) for both legends, as we're creating them manually.
# Use BATCH_ORDER directly to ensure consistent legend order
unique_batches_day5 = BATCH_ORDER # Use the predefined order for legends
palette_colors = sns.color_palette('viridis', n_colors=len(unique_batches_day5))
batch_color_map = {batch: color for batch, color in zip(unique_batches_day5, palette_colors)}
proxy_handles = [mpatches.Patch(color=batch_color_map[batch], label=batch) for batch in unique_batches_day5]
labels = list(unique_batches_day5) # Labels for the legend

# 1. Create the horizontal legend under the title (figure-level).
unique_batches_count_day5 = len(unique_batches_day5)
legend_horizontal = g.fig.legend(handles=proxy_handles, labels=labels,
                                 loc='upper center', bbox_to_anchor=(0.5, 0.98), # Position near top-center
                                 ncol=unique_batches_count_day5, title='BATCH', frameon=False,
                                 fontsize='small') # Unique property for separate legend instance

# 2. Create the second 'default' style legend in the top-right corner (figure-level).
legend_top_right = g.fig.legend(handles=proxy_handles, labels=labels,
                                loc='upper right', bbox_to_anchor=(0.99, 0.95), # Position near top-right of figure
                                title='BATCH', frameon=True,
                                fontsize='medium') # Unique property for separate legend instance

# Adjust layout to make space for the title and both figure-level legends.
# The rect argument defines the bounding box for the subplots.
# (left, bottom, right, top). We leave space at the top (0.9).
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

### Circularity compersion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches # For creating proxy artists

# Create a catplot to generate box plots comparing batches across all days
g = sns.catplot(
    data=df, # Use the full DataFrame to include all days
    x='Condition',   # Conditions on x-axis
    y='Circ.',
    hue='BATCH',     # Batches as hue for comparison within conditions
    hue_order=BATCH_ORDER, # Explicitly set the order of batches
    row='DAY',       # Arrange days vertically in separate rows
    kind='box',
    palette='viridis', # Apply palette for hue
    height=3,        # Adjusted height for multiple rows
    aspect=6.25,      # Adjusted aspect ratio for wider plots (was 2.5, now doubled)
    boxprops={'edgecolor': 'black', 'linewidth': 0.5}, # Consistent box properties
    width=0.75,
    legend=False     # Suppress the default FacetGrid legend
)

# Set common properties for all facets
g.set_axis_labels('Condition', 'Circularity')
g.set_titles(row_template='{row_name}') # Title for each row (Day)
g.set(ylim=CIRCULARITY_BOXPLOT_YLIM)

# Add grid to each subplot
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75)
    ax.tick_params(axis='x', rotation=45) # Rotate x-axis labels if they overlap

# Add a main title to the entire figure, positioned slightly higher
g.fig.suptitle('Circularity Distribution by Day, Condition, and Batch', y=1.03)

# Prepare proxy artists (patches) for both legends, as we're creating them manually.
# Use BATCH_ORDER directly to ensure consistent legend order
unique_batches_for_legend = BATCH_ORDER # Use the predefined order for legends
palette_colors = sns.color_palette('viridis', n_colors=len(unique_batches_for_legend))
batch_color_map = {batch: color for batch, color in zip(unique_batches_for_legend, palette_colors)}
proxy_handles = [mpatches.Patch(color=batch_color_map[batch], label=batch) for batch in unique_batches_for_legend]
labels = list(unique_batches_for_legend) # Labels for the legend

# 1. Create the horizontal legend under the title (figure-level).
unique_batches_count = len(unique_batches_for_legend)
legend_horizontal = g.fig.legend(handles=proxy_handles, labels=labels,
                                 loc='upper center', bbox_to_anchor=(0.5, 0.98), # Position near top-center
                                 ncol=unique_batches_count, title='BATCH', frameon=False,
                                 fontsize='small') # Unique property for separate legend instance

# 2. Create the second 'default' style legend in the top-right corner (figure-level).
legend_top_right = g.fig.legend(handles=proxy_handles, labels=labels,
                                loc='upper right', bbox_to_anchor=(0.99, 0.95), # Position near top-right of figure
                                title='BATCH', frameon=True,
                                fontsize='medium') # Unique property for separate legend instance

# Adjust layout to make space for the title and both figure-level legends.
# The rect argument defines the bounding box for the subplots.
# (left, bottom, right, top). We leave space at the top (0.9).
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

###Circularity but only 1 day

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches # For creating proxy artists

# Filter the DataFrame to include only 'DAY5'
df_day5 = df[df['DAY'] == 'DAY5'].copy()

# Create a catplot to generate box plots comparing batches for DAY5
g = sns.catplot(
    data=df_day5,
    x='Condition',   # Conditions on x-axis
    y='Circ.',
    hue='BATCH',     # Batches as hue for comparison within conditions
    kind='box',
    palette='viridis', # Apply palette for hue
    height=7,        # Height of the single facet
    aspect=3.0,      # Aspect ratio for a wider single plot
    boxprops={'edgecolor': 'black', 'linewidth': 0.5}, # Consistent box properties
    width=0.75,
    legend=False     # Suppress the default FacetGrid legend
)

# Set common properties for the single facet
g.set_axis_labels('Condition', 'Circularity')
g.set(ylim=CIRCULARITY_BOXPLOT_YLIM)

# Add grid to the subplot
for ax in g.axes.flat:
    ax.grid(True, linestyle='--', alpha=0.75)
    ax.tick_params(axis='x', rotation=45) # Rotate x-axis labels if they overlap

# Add a main title to the entire figure, positioned slightly higher
g.fig.suptitle('ImageJ: Circularity Distribution for Day 5 by Condition and Batch', y=1.03)

# Prepare proxy artists (patches) for both legends, as we're creating them manually.
unique_batches_day5 = df_day5['BATCH'].dropna().unique()
palette_colors = sns.color_palette('viridis', n_colors=len(unique_batches_day5))
batch_color_map = {batch: color for batch, color in zip(unique_batches_day5, palette_colors)}
proxy_handles = [mpatches.Patch(color=batch_color_map[batch], label=batch) for batch in unique_batches_day5]
labels = list(unique_batches_day5) # Labels for the legend

# 1. Create the horizontal legend under the title (figure-level).
unique_batches_count_day5 = len(unique_batches_day5)
legend_horizontal = g.fig.legend(handles=proxy_handles, labels=labels,
                                 loc='upper center', bbox_to_anchor=(0.5, 0.98), # Position near top-center
                                 ncol=unique_batches_count_day5, title='BATCH', frameon=False,
                                 fontsize='small') # Unique property for separate legend instance

# 2. Create the second 'default' style legend in the top-right corner (figure-level).
legend_top_right = g.fig.legend(handles=proxy_handles, labels=labels,
                                loc='upper right', bbox_to_anchor=(0.99, 0.95), # Position near top-right of figure
                                title='BATCH', frameon=True,
                                fontsize='medium') # Unique property for separate legend instance

# Adjust layout to make space for the title and both figure-level legends.
# The rect argument defines the bounding box for the subplots.
# (left, bottom, right, top). We leave space at the top (0.9).
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

###Percentage Size Reduction vs. NEG (Diameter)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches

# Ensure BATCH_ORDER and CONDITION_ORDER are defined
if 'BATCH_ORDER' not in globals():
    BATCH_ORDER = ['STD-MAN-R6', 'STD-ON-R6', 'ZEI-ON-R6']
    print(f"DEBUG: BATCH_ORDER was not defined, initialized to: {BATCH_ORDER}")
if 'CONDITION_ORDER' not in globals():
    CONDITION_ORDER = ['DS', 'HS', 'PEG', 'PVA', 'DSPEG', 'DSPVA', 'HSPEG', 'NEG']
    print(f"DEBUG: CONDITION_ORDER was not defined, initialized to: {CONDITION_ORDER}")

# Calculate mean diameter for each BATCH, DAY, Condition group
mean_diameter_df = df.groupby(['BATCH', 'DAY', 'Condition'], observed=False)['Diameter'].mean().reset_index()

# Initialize a list to store percentage reduction data
percentage_reduction_data = []

# Iterate through unique BATCH and DAY combinations
for batch in BATCH_ORDER:
    for day in df['DAY'].cat.categories:
        # Get the mean diameter for 'NEG' condition for the current batch and day
        neg_mean_diameter = mean_diameter_df[
            (mean_diameter_df['BATCH'] == batch) &
            (mean_diameter_df['DAY'] == day) &
            (mean_diameter_df['Condition'] == 'NEG')
        ]['Diameter'].iloc[0] if not mean_diameter_df[
            (mean_diameter_df['BATCH'] == batch) &
            (mean_diameter_df['DAY'] == day) &
            (mean_diameter_df['Condition'] == 'NEG')
        ].empty else None

        if neg_mean_diameter is not None and neg_mean_diameter != 0:
            # Iterate through all other conditions for the current batch and day
            for condition in CONDITION_ORDER:
                if condition == 'NEG': # Skip NEG vs NEG comparison
                    continue

                current_condition_mean_diameter = mean_diameter_df[
                    (mean_diameter_df['BATCH'] == batch) &
                    (mean_diameter_df['DAY'] == day) &
                    (mean_diameter_df['Condition'] == condition)
                ]['Diameter'].iloc[0] if not mean_diameter_df[
                    (mean_diameter_df['BATCH'] == batch) &
                    (mean_diameter_df['DAY'] == day) &
                    (mean_diameter_df['Condition'] == condition)
                ].empty else None

                if current_condition_mean_diameter is not None:
                    # Calculate percentage reduction
                    # (NEG - Condition) / NEG * 100
                    percentage_reduction = ((neg_mean_diameter - current_condition_mean_diameter) / neg_mean_diameter) * 100
                    percentage_reduction_data.append({
                        'BATCH': batch,
                        'DAY': day,
                        'Condition': condition,
                        'Percentage_Reduction_vs_NEG': percentage_reduction
                    })

# Create a DataFrame from the collected data
percentage_reduction_df = pd.DataFrame(percentage_reduction_data)

# Ensure 'DAY' and 'Condition' are categorical for correct ordering in plot
percentage_reduction_df['DAY'] = pd.Categorical(percentage_reduction_df['DAY'], categories=df['DAY'].cat.categories, ordered=True)
percentage_reduction_df['Condition'] = pd.Categorical(percentage_reduction_df['Condition'],
                                                     categories=[c for c in CONDITION_ORDER if c != 'NEG'], ordered=True)

# Plotting the percentage reduction
if not percentage_reduction_df.empty:
    g = sns.catplot(
        data=percentage_reduction_df,
        x='Condition',          # Conditions on x-axis
        y='Percentage_Reduction_vs_NEG', # Y-axis is the calculated percentage reduction
        hue='BATCH',            # Batches as hue for comparison
        hue_order=BATCH_ORDER,  # Explicitly set batch order
        col='DAY',              # Arrange days horizontally
        col_wrap=2,             # Wrap columns to create a 2x2 grid
        kind='bar',             # Bar plot type
        palette='viridis',      # Apply palette for hue
        height=3.5,             # Height of each facet
        aspect=1.5,             # Aspect ratio of each facet
        errorbar=None,          # No error bars for this plot as it's based on means
        width=0.75,             # Bar width
        legend=False            # Suppress default legend
    )

    g.set_axis_labels('Condition', 'Diameter Reduction vs NEG (%)')
    g.set_titles(col_template='Day: {col_name}')
    g.fig.suptitle('Diameter Percentage Reduction vs. NEG by Day and Batch', y=1.03)

    # Add grid and rotate x-axis labels
    for ax in g.axes.flat:
        ax.grid(True, linestyle='--', alpha=0.75)
        ax.tick_params(axis='x', rotation=45)
        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8) # Add a horizontal line at 0%

    # Manual legend for BATCH
    unique_batches_for_legend = BATCH_ORDER
    palette_colors = sns.color_palette('viridis', n_colors=len(unique_batches_for_legend))
    batch_color_map = {batch: color for batch, color in zip(unique_batches_for_legend, palette_colors)}
    proxy_handles = [mpatches.Patch(color=batch_color_map[batch], label=batch) for batch in unique_batches_for_legend]
    labels = list(unique_batches_for_legend)

    g.fig.legend(handles=proxy_handles, labels=labels,
                  loc='upper center', bbox_to_anchor=(0.5, 0.98),
                  ncol=len(unique_batches_for_legend), title='BATCH', frameon=False,
                  fontsize='small')

    plt.tight_layout(rect=[0, 0, 1, 0.9])
    plt.show()
else:
    print("No percentage reduction data to plot.")

##Statistics

###Bland-altman with Holm correction

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

BATCH_ORDER = ['STD-MAN-R6', 'STD-ON-R6', 'ZEI-ON-R6']
REFERENCE  = 'STD-MAN-R6'
DAY_ORDER  = ['DAY2', 'DAY3', 'DAY4', 'DAY5']

# ---------------------------------------------------------------
# 1. Aggregate to WELL level (one summary per BATCH x DAY x Condition)
#    This is the experimental unit, NOT individual aggregates.
# ---------------------------------------------------------------
well = (df.groupby(['BATCH', 'DAY', 'Condition'], observed=False)['Diameter']
         .agg(mean='mean', median='median', n='count').reset_index())

def pct_above(d, thr=400):
    d = d.dropna()
    return 100 * np.mean(d >= thr) if len(d) else np.nan
p400 = (df.groupby(['BATCH','DAY','Condition'], observed=False)['Diameter']
          .apply(pct_above).reset_index(name='pct_above_400'))
well = well.merge(p400, on=['BATCH','DAY','Condition'])

# pivot to wide: one row per (DAY, Condition), columns per method
def pivot(col):
    t = well.pivot_table(index=['DAY','Condition'], columns='BATCH',
                         values=col, observed=False).reset_index()
    t.columns.name = None
    return t
mean_w  = pivot('mean')
p400_w  = pivot('pct_above_400')

# keep only matched rows (all three methods present)
mean_w = mean_w.dropna(subset=BATCH_ORDER)
p400_w = p400_w.dropna(subset=BATCH_ORDER)
print(f"Matched well-level pairs: n = {len(mean_w)}")

# ---------------------------------------------------------------
# 2. Bland-Altman + CCC for each pair (paired by DAY x Condition)
# ---------------------------------------------------------------
def bland_altman(a, b, label_a, label_b):
    d = a - b
    m = (a + b) / 2
    bias = d.mean()
    sd   = d.std(ddof=1)
    loa  = 1.96 * sd
    # 95% CI for the bias
    se   = sd / np.sqrt(len(d))
    ci   = stats.t.interval(0.95, len(d)-1, loc=bias, scale=se)
    # Concordance correlation coefficient (Lin)
    cov  = np.cov(a, b, ddof=1)[0,1]
    ccc  = 2*cov / (a.var(ddof=1) + b.var(ddof=1) + (a.mean()-b.mean())**2)
    # paired test (normal diffs -> t-test; otherwise Wilcoxon)
    t, p_t = stats.ttest_rel(a, b)
    w, p_w = stats.wilcoxon(a, b)
    print(f"\n{label_a} vs {label_b}")
    print(f"  bias (mean diff) = {bias:.2f}  95% CI [{ci[0]:.2f}, {ci[1]:.2f}]")
    print(f"  limits of agreement = ±{loa:.2f}")
    print(f"  CCC (Lin) = {ccc:.3f}")
    print(f"  paired t-test: t={t:.2f}, p={p_t:.3g}")
    print(f"  Wilcoxon signed-rank: p={p_w:.3g}")
    return bias, loa, ccc, p_t, p_w, m, d

pairs = [('STD-MAN-R6','STD-ON-R6'), ('STD-MAN-R6','ZEI-ON-R6'), ('STD-ON-R6','ZEI-ON-R6')]
rows = []
fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4.5))
for ax, (a_id, b_id) in zip(axes, pairs):
    res = bland_altman(mean_w[a_id].values, mean_w[b_id].values, a_id, b_id)
    bias, loa, ccc, p_t, p_w, m, d = res
    rows.append({'pair': f'{a_id} vs {b_id}', 'bias':bias, 'LoA':loa, 'CCC':ccc, 'p_paired_t':p_t, 'p_wilcoxon':p_w})
    ax.scatter(m, d, alpha=0.6)
    ax.axhline(bias, color='red', ls='-')
    ax.axhline(bias+loa, color='red', ls='--')
    ax.axhline(bias-loa, color='red', ls='--')
    ax.set_xlabel('Mean of two methods (µm)'); ax.set_ylabel(f'{a_id} - {b_id} (µm)')
    ax.set_title(f'{a_id} vs {b_id}\nCCC={ccc:.2f}, bias={bias:.1f}µm')
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------
# 3. Holm correction across the three pairwise paired t-tests
# ---------------------------------------------------------------
ba = pd.DataFrame(rows)
pvals = ba['p_paired_t'].values
order = np.argsort(pvals)
adj = np.empty(len(pvals))
for i, idx in enumerate(order):
    adj[idx] = min(1, pvals[idx] * (len(pvals) - i))
adj = np.maximum.accumulate(adj[order])[np.argsort(order)]
ba['p_holm'] = adj
print("\n--- Bland-Altman summary (Holm-corrected paired t-test p-values) ---")
print(ba.round(4).to_string(index=False))

###Kolmogorov-Smirnov Tests

In [ ]:
import itertools
from scipy.stats import ks_2samp
from collections import defaultdict

# Ensure BATCH column is categorical before accessing .cat accessor
# Assuming BATCH_ORDER is defined in a preceding cell or globally available.
# If BATCH_ORDER is not globally defined, it would need to be defined here.
if 'BATCH_ORDER' in globals() and not pd.api.types.is_categorical_dtype(df['BATCH']):
    df['BATCH'] = pd.Categorical(df['BATCH'], categories=BATCH_ORDER, ordered=True)

# --- Diameter Comparisons ---
print("\n--- Pairwise Kolmogorov-Smirnov Tests: Batches on the Same Day and Condition (Diameter) ---")

unique_days = df['DAY'].cat.categories
unique_conditions = df['Condition'].cat.categories
unique_batches = df['BATCH'].cat.categories

total_comparisons_dia = 0
significant_comparisons_dia = 0
non_significant_comparisons_dia = 0
batch_deviations_count_dia = defaultdict(int)
significant_pair_counts_dia = defaultdict(int) # New: Track significant comparisons per pair

for day in unique_days:
    print(f"\nDay: {day}")
    for condition in unique_conditions:
        print(f"  Condition: {condition}")

        # Get batches present for the current day and condition
        batches_on_day_condition = df[(df['DAY'] == day) & (df['Condition'] == condition)]['BATCH'].unique().tolist()
        batches_on_day_condition.sort() # Sort for consistent output

        if len(batches_on_day_condition) < 2:
            print(f"    Not enough batches ({len(batches_on_day_condition)}) for pairwise comparison on {day}, {condition}.")
            continue

        for batch1, batch2 in itertools.combinations(batches_on_day_condition, 2):
            total_comparisons_dia += 1
            data_batch1 = df[(df['DAY'] == day) & (df['Condition'] == condition) & (df['BATCH'] == batch1)]['Diameter'].dropna()
            data_batch2 = df[(df['DAY'] == day) & (df['Condition'] == condition) & (df['BATCH'] == batch2)]['Diameter'].dropna()

            if len(data_batch1) > 1 and len(data_batch2) > 1: # Ensure enough data points for the test
                statistic_pair, p_value_pair = ks_2samp(data_batch1, data_batch2)
                print(f"    {batch1} vs {batch2}: KS Stat = {statistic_pair:.4f}, P-value = {p_value_pair:.2e}", end="")
                if p_value_pair < 0.05:
                    significant_comparisons_dia += 1
                    batch_deviations_count_dia[batch1] += 1
                    batch_deviations_count_dia[batch2] += 1
                    # New: Increment count for this specific batch pair
                    pair_key = tuple(sorted((batch1, batch2))) # Use a sorted tuple for consistent key
                    significant_pair_counts_dia[pair_key] += 1
                    print(" (Significant)")
                else:
                    non_significant_comparisons_dia += 1
                    print(" (Not significant)")
            else:
                non_significant_comparisons_dia += 1 # Count as non-significant if not enough data for test
                print(f"    Not enough data for {batch1} or {batch2} on {day}, {condition} for KS test. (Considered Not significant)")
print("-" * 50)

print("\n--- Summary of Pairwise KS Tests (Diameter) ---")
print(f"Total comparisons made: {total_comparisons_dia}")
print(f"Significant comparisons (p < 0.05): {significant_comparisons_dia}")
print(f"Non-significant comparisons (p >= 0.05): {non_significant_comparisons_dia}")

# New: Print significant comparison counts for each pair
if significant_pair_counts_dia:
    print("Significant comparisons by batch pair:")
    for pair, count in sorted(significant_pair_counts_dia.items()):
        print(f"  {pair[0]} vs {pair[1]}: {count} significant comparisons")

if batch_deviations_count_dia:
    most_deviated_batch_dia = max(batch_deviations_count_dia, key=batch_deviations_count_dia.get)
    print(f"Batch that mostly deviated (involved in most significant comparisons): {most_deviated_batch_dia} ({batch_deviations_count_dia[most_deviated_batch_dia]} times)")
else:
    print("No significant deviations found between batches for Diameter.")
print("-" * 50)

# --- Circularity Comparisons ---
print("\n--- Pairwise Kolmogorov-Smirnov Tests: Batches on the Same Day and Condition (Circularity) ---")

total_comparisons_circ = 0
significant_comparisons_circ = 0
non_significant_comparisons_circ = 0
batch_deviations_count_circ = defaultdict(int)
significant_pair_counts_circ = defaultdict(int) # New: Track significant comparisons per pair

for day in unique_days:
    print(f"\nDay: {day}")
    for condition in unique_conditions:
        print(f"  Condition: {condition}")

        batches_on_day_condition = df[(df['DAY'] == day) & (df['Condition'] == condition)]['BATCH'].unique().tolist()
        batches_on_day_condition.sort() # Sort for consistent output

        if len(batches_on_day_condition) < 2:
            print(f"    Not enough batches ({len(batches_on_day_condition)}) for pairwise comparison on {day}, {condition}.")
            continue

        for batch1, batch2 in itertools.combinations(batches_on_day_condition, 2):
            total_comparisons_circ += 1
            data_batch1 = df[(df['DAY'] == day) & (df['Condition'] == condition) & (df['BATCH'] == batch1)]['Circ.'].dropna()
            data_batch2 = df[(df['DAY'] == day) & (df['Condition'] == condition) & (df['BATCH'] == batch2)]['Circ.'].dropna()

            if len(data_batch1) > 1 and len(data_batch2) > 1: # Ensure enough data points for the test
                statistic_pair, p_value_pair = ks_2samp(data_batch1, data_batch2)
                print(f"    {batch1} vs {batch2}: KS Stat = {statistic_pair:.4f}, P-value = {p_value_pair:.2e}", end="")
                if p_value_pair < 0.05:
                    significant_comparisons_circ += 1
                    batch_deviations_count_circ[batch1] += 1
                    batch_deviations_count_circ[batch2] += 1
                    # New: Increment count for this specific batch pair
                    pair_key = tuple(sorted((batch1, batch2))) # Use a sorted tuple for consistent key
                    significant_pair_counts_circ[pair_key] += 1
                    print(" (Significant)")
                else:
                    non_significant_comparisons_circ += 1
                    print(" (Not significant)")
            else:
                non_significant_comparisons_circ += 1 # Count as non-significant if not enough data for test
                print(f"    Not enough data for {batch1} or {batch2} on {day}, {condition} for KS test. (Considered Not significant)")
print("-" * 50)

print("\n--- Summary of Pairwise KS Tests (Circularity) ---")
print(f"Total comparisons made: {total_comparisons_circ}")
print(f"Significant comparisons (p < 0.05): {significant_comparisons_circ}")
print(f"Non-significant comparisons (p >= 0.05): {non_significant_comparisons_circ}")

# New: Print significant comparison counts for each pair
if significant_pair_counts_circ:
    print("Significant comparisons by batch pair:")
    for pair, count in sorted(significant_pair_counts_circ.items()):
        print(f"  {pair[0]} vs {pair[1]}: {count} significant comparisons")

if batch_deviations_count_circ:
    most_deviated_batch_circ = max(batch_deviations_count_circ, key=batch_deviations_count_circ.get)
    print(f"Batch that mostly deviated (involved in most significant comparisons): {most_deviated_batch_circ} ({batch_deviations_count_circ[most_deviated_batch_circ]} times)")
else:
    print("No significant deviations found between batches for Circularity.")
print("-" * 50)

###Kruskal-Wallis H-test

In [ ]:
import pandas as pd
from scipy.stats import kruskal
import itertools

# Ensure BATCH column is categorical, if not already
# This check is robust as 'BATCH_ORDER' should be defined globally by now
# Updated to use isinstance(dtype, pd.CategoricalDtype) to address DeprecationWarning
if 'BATCH_ORDER' in globals() and not isinstance(df['BATCH'].dtype, pd.CategoricalDtype):
    df['BATCH'] = pd.Categorical(df['BATCH'], categories=BATCH_ORDER, ordered=True)

print("--- Kruskal-Wallis H-test: Comparing all Batches on the Same Day and Condition (Diameter) ---")

unique_days = df['DAY'].cat.categories
unique_conditions = df['Condition'].cat.categories

for day in unique_days:
    print(f"\nDay: {day}")
    for condition in unique_conditions:
        print(f"  Condition: {condition}")

        # Collect Diameter data for each batch for the current day and condition
        batch_data_for_test = []
        available_batches = []

        for batch in BATCH_ORDER: # Use the predefined order for batches
            data = df[(df['DAY'] == day) & (df['Condition'] == condition) & (df['BATCH'] == batch)]['Diameter'].dropna()
            if len(data) > 1: # Kruskal-Wallis requires at least 2 data points per group
                batch_data_for_test.append(data.values)
                available_batches.append(batch)

        if len(batch_data_for_test) >= 2: # Kruskal-Wallis needs at least two groups to compare
            # Perform the Kruskal-Wallis H-test
            h_statistic, p_value = kruskal(*batch_data_for_test)

            batches_compared_str = ', '.join(available_batches)
            print(f"    Comparing [{batches_compared_str}]: H-statistic = {h_statistic:.4f}, P-value = {p_value:.2e}", end="")
            if p_value < 0.05:
                print(" (Significant difference between at least two groups)")
            else:
                print(" (No significant difference between groups)")
        elif len(batch_data_for_test) == 1:
            print(f"    Only one batch ({available_batches[0]}) has sufficient data for comparison on {day}, {condition}. Skipping Kruskal-Wallis test.")
        else:
            print(f"    Not enough batches with sufficient data for comparison on {day}, {condition}. Skipping Kruskal-Wallis test.")
print("-" * 50)


###Simple pair mean diameter percentage difference

In [ ]:
import itertools
import pandas as pd

# Ensure BATCH, DAY, Condition columns are categorical
# Use BATCH_ORDER, CONDITION_ORDER, and sorted_days (if defined) for categories
if 'BATCH_ORDER' in globals() and not isinstance(df['BATCH'].dtype, pd.CategoricalDtype):
    df['BATCH'] = pd.Categorical(df['BATCH'], categories=BATCH_ORDER, ordered=True)

# Assuming CONDITION_ORDER is defined in a preceding cell
if 'CONDITION_ORDER' in globals() and not isinstance(df['Condition'].dtype, pd.CategoricalDtype):
    df['Condition'] = pd.Categorical(df['Condition'], categories=CONDITION_ORDER, ordered=True)

# Assuming sorted_days is defined in a preceding cell (e.g., in the data processing cell)
# If not, generate it here based on unique values to ensure 'DAY' is categorical
if 'sorted_days' in globals() and not isinstance(df['DAY'].dtype, pd.CategoricalDtype):
    df['DAY'] = pd.Categorical(df['DAY'], categories=sorted_days, ordered=True)
elif not isinstance(df['DAY'].dtype, pd.CategoricalDtype): # Fallback if sorted_days not globally defined
    unique_days_raw = df['DAY'].dropna().unique()
    def natural_sort_key(s):
        return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', str(s))]
    sorted_days = sorted(unique_days_raw, key=natural_sort_key)
    df['DAY'] = pd.Categorical(df['DAY'], categories=sorted_days, ordered=True)


print("--- Pairwise Mean Diameter Percentage Difference Between Batches ---")

unique_days = df['DAY'].cat.categories
unique_conditions = df['Condition'].cat.categories
unique_batches = df['BATCH'].cat.categories

for day in unique_days:
    print(f"\nDay: {day}")
    for condition in unique_conditions:
        print(f"  Condition: {condition}")

        # Get mean diameters for each batch for the current day and condition
        mean_diameters = {}
        batches_present = []

        for batch in unique_batches:
            data = df[(df['DAY'] == day) & (df['Condition'] == condition) & (df['BATCH'] == batch)]['Diameter'].dropna()
            if len(data) > 0:
                mean_diameters[batch] = data.mean()
                batches_present.append(batch)

        if len(batches_present) < 2:
            print(f"    Not enough batches with data for comparison on {day}, {condition}.")
            continue

        # Perform pairwise comparisons
        for batch1, batch2 in itertools.combinations(batches_present, 2):
            mean1 = mean_diameters[batch1]
            mean2 = mean_diameters[batch2]

            if pd.isna(mean1) or pd.isna(mean2):
                print(f"    {batch1} vs {batch2}: Not enough data for mean diameter calculation.")
                continue

            # Calculate percentage difference relative to batch2
            # This shows how much batch1's mean diameter differs from batch2's
            percentage_diff_1_vs_2 = ((mean1 - mean2) / mean2) * 100
            print(f"    {batch1} mean vs {batch2} mean: {percentage_diff_1_vs_2:.2f}% (Batch1 mean = {mean1:.2f}, Batch2 mean = {mean2:.2f})")

            # Calculate percentage difference relative to batch1
            percentage_diff_2_vs_1 = ((mean2 - mean1) / mean1) * 100
            print(f"    {batch2} mean vs {batch1} mean: {percentage_diff_2_vs_1:.2f}% (Batch2 mean = {mean2:.2f}, Batch1 mean = {mean1:.2f})")

print("-" * 50)


#DELETE FILES

In [ ]:
import os
import glob
import re

# Assume files are uploaded to /content/
target_directory = '/content/'

# List all files in the target directory (refresh the list)
all_files = os.listdir(target_directory)

# Filter for files that look like copies: have ' (number)' before '.csv'
copy_files_to_delete = []
for filename in all_files:
    # Updated regex to match ' (N).csv' or '(N).csv' at the end of the filename
    # The previous regex required a space before '(', but the uploaded files did not have it.
    # Also, glob.glob was not used, so the all_files list needed to be updated.
    if re.search(r'\(\d+\)\.csv$', filename):
        copy_files_to_delete.append(os.path.join(target_directory, filename))

if copy_files_to_delete:
    print(f"Deleting {len(copy_files_to_delete)} copy files:")
    for f in copy_files_to_delete:
        try:
            os.remove(f)
            print(f"  - Removed: {f}")
        except OSError as e:
            print(f"  - Error deleting {f}: {e}")
else:
    print("No copy files found to delete with the pattern ' (N).csv' or '(N).csv'.")